# 03b — BTS Flight Data Acquisition (Problem C)

Download BTS On-Time Reporting Carrier performance data.  
These are the **control flights** (uneventful) for our case-control study.

**Data source:** https://transtats.bts.gov/PREZIP/  
**Scope:** 2018–2020 (POC, ~18M flights)  
**Output:** `data/raw/bts_flights/` (one parquet per year)

In [ ]:
import os
from pathlib import Path

# Find project root (contains pyproject.toml)
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import requests
import zipfile
import io
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BTS_RAW = Path('data/raw/bts_flights')
BTS_RAW.mkdir(parents=True, exist_ok=True)

YEAR_START = 2018
YEAR_END   = 2020

# Columns we need (keeps download small)
KEEP_COLS = [
    'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM',
    'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR',
    'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR',
    'DEP_TIME', 'DEP_DELAY', 'ARR_TIME', 'ARR_DELAY',
    'CANCELLED', 'DIVERTED', 'DISTANCE',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY',
]

print(f'POC scope: {YEAR_START}–{YEAR_END}')
print(f'Keeping {len(KEEP_COLS)} columns per flight')

## 1. Download BTS Monthly ZIPs

BTS provides pre-zipped CSVs at:  
`https://transtats.bts.gov/PREZIP/On_Time_Reporting_..._{YEAR}_{MONTH}.zip`

Each file is ~20-30MB compressed, ~150MB CSV.

In [ ]:
BASE_URL = 'https://transtats.bts.gov/PREZIP'
FILE_TEMPLATE = 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip'

def download_bts_month(year, month, out_dir):
    """Download one month of BTS on-time data. Returns path to extracted CSV."""
    filename = FILE_TEMPLATE.format(year=year, month=month)
    zip_path = out_dir / filename
    
    # Check if parquet already exists
    parquet_path = out_dir / f'bts_{year}_{month:02d}.parquet'
    if parquet_path.exists():
        return parquet_path, 'cached'
    
    url = f'{BASE_URL}/{filename}'
    print(f'  Downloading {filename}...', end=' ')
    
    try:
        resp = requests.get(url, stream=True, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f'FAILED: {e}')
        return None, str(e)
    
    # Extract CSV from ZIP in memory
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        csv_names = [n for n in z.namelist() if n.endswith('.csv')]
        if not csv_names:
            print('No CSV in ZIP')
            return None, 'no_csv'
        
        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f, low_memory=False, usecols=lambda c: c in KEEP_COLS)
    
    # Save as parquet (much smaller)
    df.to_parquet(parquet_path, index=False)
    size_mb = parquet_path.stat().st_size / 1e6
    print(f'{len(df):,} flights, {size_mb:.1f}MB parquet')
    
    return parquet_path, 'downloaded'

print('Download function ready.')

In [ ]:
# Download all months in POC range
results = []

for year in range(YEAR_START, YEAR_END + 1):
    print(f'\n=== {year} ===')
    for month in range(1, 13):
        path, status = download_bts_month(year, month, BTS_RAW)
        results.append({'year': year, 'month': month, 'path': path, 'status': status})
        if status == 'downloaded':
            time.sleep(2)  # Be polite to BTS servers

results_df = pd.DataFrame(results)
print(f'\n\nDownload summary:')
print(results_df['status'].value_counts())

## 2. Consolidate Per-Year Parquets

In [ ]:
yearly_stats = []

for year in range(YEAR_START, YEAR_END + 1):
    year_files = sorted(BTS_RAW.glob(f'bts_{year}_*.parquet'))
    if not year_files:
        print(f'{year}: No files found')
        continue
    
    dfs = [pd.read_parquet(f) for f in year_files]
    year_df = pd.concat(dfs, ignore_index=True)
    
    # Save consolidated year file
    out = BTS_RAW / f'bts_{year}.parquet'
    year_df.to_parquet(out, index=False)
    
    yearly_stats.append({
        'year': year,
        'flights': len(year_df),
        'carriers': year_df['OP_UNIQUE_CARRIER'].nunique() if 'OP_UNIQUE_CARRIER' in year_df.columns else 0,
        'airports': year_df['ORIGIN'].nunique() if 'ORIGIN' in year_df.columns else 0,
        'size_mb': out.stat().st_size / 1e6,
    })
    print(f'{year}: {len(year_df):,} flights, {yearly_stats[-1]["carriers"]} carriers, {yearly_stats[-1]["airports"]} airports')

print(pd.DataFrame(yearly_stats).to_string(index=False))

## 3. Quick Profile

In [ ]:
# Load one year for profiling
sample_path = BTS_RAW / f'bts_{YEAR_START}.parquet'
if sample_path.exists():
    sample = pd.read_parquet(sample_path)
    print(f'Sample ({YEAR_START}): {sample.shape}')
    print(f'\nColumn fill rates:')
    for col in sample.columns:
        fill = sample[col].notna().mean() * 100
        print(f'  {col:30s} {fill:5.1f}%')
    
    print(f'\nTop 10 origin airports:')
    print(sample['ORIGIN'].value_counts().head(10))
    
    print(f'\nCancellation rate: {sample["CANCELLED"].mean()*100:.2f}%')
    print(f'Diversion rate: {sample["DIVERTED"].mean()*100:.2f}%')
else:
    print(f'{sample_path} not found. Run download cells first.')

In [ ]:
print('BTS data acquisition complete.')
print(f'Files saved to: {BTS_RAW.resolve()}')
print(f'\nTotal disk usage:')
total = sum(f.stat().st_size for f in BTS_RAW.glob('*.parquet'))
print(f'  {total / 1e6:.1f} MB in parquet files')